# Import of Data from .txt files to create Databases

Here the Pandas library is being used to get the databases of all the analyzed cases.

In [1]:
import numpy as np
import pandas as pd
import os, glob
import re
import matplotlib.pyplot as plt
# %run master_thesis_functions.ipynb 

## Import of the Summary of all measured cases

Here the summary of all the recorded cases is imported from the excel file overview_experiments.csv.

In [2]:
excel = pd.read_csv(r'/Users/santiago/Documents/Master_Thesis_Jupyter/bacteria_folders/' + 'overview_experiments.csv', sep=';')
data_overview = excel.drop(['To do', 'In report', 'Things changed'], axis=1)
data_overview = data_overview.replace(np.nan, 'None')

In [3]:
data_overview

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
0,19.04.2022,Nanomotion measurement,7740 hypermotile Cells (Control and Sensitive),OSC,None,Sensitive cells with added meropenem10ug/ml,None
1,11.05.2022,Nanomotion measurement,2139-06154 NDM5 Ecoli (Resistent Mero),10 um/ R308_73 / depth 903 um,None,Resistant cells with added meropenem 8ug/ml,Variance goes up after adding meropenem (so yo...
2,13.05.2022,Nanomotion measurement,881-61610 blAOXA-48 (Sensitive Mero),R308_73/ 10 um/ depth 903 um,None,Sensitive cells with added meropenem10ug/ml. m...,Variance after addition of meropenem went to a...
3,17.05.2022,Nanomotion measurement,880-92227 blaOXA-244 (Sensitive Mero),R308_76/ 10 um/ depth 1258 um,None,None,None
4,18.05.2022,Nanomotion measurement,1303-49082-01 NDM7 (Resistent Mero),None,None,"Resistant cells with added meropenem 8ug/ml , ...",First measurement a lot of signal then after a...
5,19.05.2022,Nanomotion measurement,1941-17928 NDM7 (Resistent Mero),R308_77/ 8 um/ depth 1258 um,None,The first measurement without the drug showed ...,"First measurement variance of 2000, this is no..."
6,20.05.2022,Nanomotion measurement,1544-15672-03 bla OXA-48 like (Sensitive Mero),R308_75/ 10 um/ depth 1258 um,100 metingen 30 s,None,Variance voor antibioticum 8 daarna naar achte...
7,14.06.2022,Optical signal measurement,No cells just LB,10 um/ depth 285 um No Graphene,None,None,None
8,21.06.2022,Optical signal measurement,No cells just LB,10 um/ depth 285 um No Graphene,15 wells 150s with normal conditions (light an...,None,All measurements showed the peak at around 1.5...
9,29.11.2022,Optical signal measurement,No cells just LB,Depth 285 um (No Graphene),None,None,None


## Import of data from all measured cases 

Here a function that is capable of importing all the .txt files in a folder is being created. The function import_data_from_folder receives as parameters the Name of the Folder to import, the Dataframe that in which you want to import the information, the Overview dataframe with the summary of all the information and the index of the row in the Overview dataframe that corresponds to the imported folder.

In [4]:
def import_data_from_folder(folder_name, new_dataframe, data_summary, index, naming_format):
    #Version 4: This funtion takes a folder_name, a previusly created new_dataframe with its corresponding headers
    #a data_summary dataframe, an index from the data_summary dataframe and a naming_format. The data_format can be
    #either 'legacy' for the previuos naming of the .txt files or 'new_naming' for the new naming format for the .txt files.
    # New in Version 4: the old dataframe.append() was replaced by ps.concat() method due to deprecation of the frame.append() method.

    path = r'/Users/santiago/Documents/Master_Thesis_Jupyter/bacteria_folders/' + folder_name + '/*.txt'
    files = glob.glob(path)
    naming_format = naming_format.lower()
    files_list = []
    
    if naming_format == 'legacy':
        for file in files:
            file_name = file.replace('/Users/santiago/Documents/Master_Thesis_Jupyter/bacteria_folders/' + folder_name + '/',"")
            files_list.append(file_name)

            desktop = '/Users/santiago/Documents/Master_Thesis_Jupyter/bacteria_folders/' + folder_name
            filePath = os.path.join(desktop, file)

            file_test = open(filePath)
            df_1 = pd.read_csv(file_test, header=None)

            new_row = {new_dataframe.columns[0]:[df_1[0].values[-1]], 
                       new_dataframe.columns[1]:[df_1[1].values], 
                       new_dataframe.columns[2]:[file_name],
                       new_dataframe.columns[3]:[data_summary['What'][index]], 
                       new_dataframe.columns[4]:[data_summary['Strain'][index]],
                       new_dataframe.columns[5]:[data_summary['Chip'][index]],
                       new_dataframe.columns[6]:[data_summary['Drums'][index]]}
            new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True) 

    elif naming_format == 'new_naming':  
        for file in files:
            file_name = file.replace('/Users/santiago/Documents/Master_Thesis_Jupyter/bacteria_folders/' + folder_name + '/',"")
            files_list.append(file_name)

            desktop = '/Users/santiago/Documents/Master_Thesis_Jupyter/bacteria_folders/' + folder_name
            filePath = os.path.join(desktop, file)

            file_test = open(filePath)
            df_1 = pd.read_csv(file_test, header=None)

            headings_from_file = file_name.split('_')
            new_row = {new_dataframe.columns[0]:[df_1[0].values[-1]],
                       new_dataframe.columns[1]:[df_1[1].values],
                       new_dataframe.columns[2]:[headings_from_file[0].capitalize()],
                       new_dataframe.columns[3]:[headings_from_file[1].capitalize()],
                       new_dataframe.columns[4]:[headings_from_file[2].capitalize()],
                       new_dataframe.columns[5]:[headings_from_file[3].capitalize()],
                       new_dataframe.columns[6]:[headings_from_file[4].split('ug')[0]],
                       new_dataframe.columns[7]:[re.split(r'\D+',headings_from_file[5])[0]], #the (r'\D+') is for excluding any character that is not a number
                       new_dataframe.columns[8]:[headings_from_file[6].split('.')[0]]} # .split('.')[0] is for only taking the drum number without '.txt'
            new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True) 

    return new_dataframe


def dataframe_to_arrays(dataframe_column):
    signals_list = []
    for element in dataframe_column:
        signals_list.append(element)
    signals_array = np.array(signals_list)
    return signals_array

## Import of Empty Cavities without graphene layer (Case 7, 8, 9, 28 and 29)

In [5]:
data_overview.iloc[[7,8]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
7,14.06.2022,Optical signal measurement,No cells just LB,10 um/ depth 285 um No Graphene,None,None,None
8,21.06.2022,Optical signal measurement,No cells just LB,10 um/ depth 285 um No Graphene,15 wells 150s with normal conditions (light an...,None,All measurements showed the peak at around 1.5...


In [6]:
empty_drums_df = pd.DataFrame(columns=['Time [s]', 'Signal','File Name', 'What', 'Strain', 'Chip','Drums'])
empty_drums_df = import_data_from_folder('2022.06.14', empty_drums_df, data_overview, index=7, naming_format = 'legacy')
empty_drums_df = import_data_from_folder('2022.06.21', empty_drums_df, data_overview, index=8, naming_format = 'legacy')

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [7]:
empty_drums_30_df = empty_drums_df[empty_drums_df['File Name'].str.contains('30S')]
# empty_drums_60_df = empty_drums_df[empty_drums_df['File Name'].str.contains('60S')]
# empty_drums_150_df = empty_drums_df[empty_drums_df['File Name'].str.contains('150S')]
# empty_drums_300_df = empty_drums_df[empty_drums_df['File Name'].str.contains('300S')]
# empty_drums_600_df = empty_drums_df[empty_drums_df['File Name'].str.contains('600S')]

In [8]:
empty_drums_30 = dataframe_to_arrays(dataframe_column = empty_drums_30_df['Signal'])
# empty_drums_60 = dataframe_to_arrays(dataframe_column = empty_drums_60_df['Signal'])
# empty_drums_150 = dataframe_to_arrays(dataframe_column = empty_drums_150_df['Signal'])
# empty_drums_300 = dataframe_to_arrays(dataframe_column = empty_drums_300_df['Signal'])
# empty_drums_600 = dataframe_to_arrays(dataframe_column = empty_drums_600_df['Signal'])

### Second batch of Empty Cavities without graphene layer
This has been done because the first batch of Empty Cavities data was data that was not using the new naming convention for the measuremet cases. Therefore, the empty_drums_30_df corresponds to the first 100 elements imported without the new naming, and the empty_drums_30_2 is the dataframe used to import Empty Cavities data with the new naming.

In [9]:
data_overview.iloc[[9,28,29]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
9,29.11.2022,Optical signal measurement,No cells just LB,Depth 285 um (No Graphene),None,None,None
28,03.03.2023,Optical signal measurement,No cells just LB,Depth 285 um (No Graphene),None,None,None
29,06.03.2023,Optical signal measurement,No cells just LB,Depth 285 um (No Graphene),None,None,None


In [10]:
empty_drums_30_2_df = pd.DataFrame(columns=['Time [s]', 'Signal', 'Chip Name', 'Species','Strain', 'Antibiotic', 'Concentration [\u03BCg]', 'Time [min]', 'Drum'])
empty_drums_30_2_df = import_data_from_folder('2022.11.29-LB control', empty_drums_30_2_df, data_overview, index=9, naming_format = 'new_naming')
empty_drums_30_2_df = import_data_from_folder('2023.03.03', empty_drums_30_2_df, data_overview, index=28, naming_format = 'new_naming') #before 12
empty_drums_30_2_df = import_data_from_folder('2023.03.06', empty_drums_30_2_df, data_overview, index=29, naming_format = 'new_naming') #before 13
empty_drums_30_2 = dataframe_to_arrays(dataframe_column = empty_drums_30_2_df['Signal'])

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


### Merging all Empty Cavities cases (30 seconds)

In [11]:
empty_drums_30 = np.vstack((empty_drums_30, empty_drums_30_2)) #merging all cases: empty LB drums no graphene (30s)

## Import of Control Empty Drums with graphene layer (No bacteria)

In [12]:
data_overview.iloc[[13,16]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
13,06.12.2022,Optical signal measurement,Drums with graphene (No bacteria),Depth 285 um (with Graphene),None,None,None
16,08.12.2022,Optical signal measurement,Drums with graphene (No bacteria),Depth 285 um (with Graphene),None,None,None


In [13]:
empty_drums_graph_df = pd.DataFrame(columns=['Time [s]', 'Signal', 'Chip Name', 'Species','Strain', 'Antibiotic', 'Concentration [\u03BCg]', 'Time [min]', 'Drum'])
empty_drums_graph_df = import_data_from_folder('2022.12.06-MH control graphene', empty_drums_graph_df, data_overview, index = 13, naming_format = 'new_naming') #before 10
empty_drums_graph_df = import_data_from_folder('2022.12.08 - MH Graphene control', empty_drums_graph_df, data_overview, index = 16, naming_format = 'new_naming') #before 11

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [14]:
empty_drums_graph_1st_batch = dataframe_to_arrays(dataframe_column = empty_drums_graph_df['Signal'])

## Import of cases with E. Coli

### Import folder '2022.04.19' with 7740 (Hypermotile Cells) (Case 0) (No Grpahene)


Import of Folder '2022.04.19' which corresponds to E. Coli 7740 (Hypermotile Cells) with and without the antibiotic Meropenem with a concentration of 10ug/ml.

Cases inside this folder: 

    - E. Coli 7740 Control
    - E. Coli 7740 with Meropenem - 10ug/ml

In [15]:
data_overview.iloc[[0]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
0,19.04.2022,Nanomotion measurement,7740 hypermotile Cells (Control and Sensitive),OSC,None,Sensitive cells with added meropenem10ug/ml,None


In [16]:
ecoli7744_df = pd.DataFrame(columns=['Time [s]', 'Signal','File Name', 'What', 'Strain', 'Chip','Drums'])
ecoli7744_df = import_data_from_folder('2022.04.19', ecoli7744_df, data_overview, index=0, naming_format = 'legacy')

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [ ]:
ecoli7744_CTR_df = ecoli7744_df[~ecoli7744_df['File Name'].str.contains('meropenem')] # the ~ is to exclude the cases with the string 'meropenem'
ecoli7744_WA_df = ecoli7744_df[ecoli7744_df['File Name'].str.contains('meropenem')]

In [18]:
ecoli7744_CTR = dataframe_to_arrays(dataframe_column = ecoli7744_CTR_df['Signal'])
ecoli7744_WA = dataframe_to_arrays(dataframe_column = ecoli7744_WA_df['Signal'])

### Merging cases of ecoli without graphene

This here correspond to E. Coli bacteria in this Python Script that were measured in an empty cavity instead of a drum with graphene on top. 

In [19]:
ecoli_CTR_no_graph = np.vstack((ecoli7744_CTR)) # merging E. coli with no graphene
ecoli_CTR_no_graph_WA = np.vstack((ecoli7744_WA)) # merging E. coli with no graphene

### Import folder '11.05.2022' with E. Coli 2139-06154 NDM5 (Resistent Mero) (Case 1)

Import of Folder '11.05.2022' which corresponds of E. Coli 2139-06154 NDM5 (Resistent Mero) with and without the antibiotic Meropenem with a concentration of 8ug/ml.

Cases inside this folder: 

    - E. Coli 2139-06154 NDM5 Control
    - E. Coli 2139-06154 NDM5 with Meropenem - 8ug/ml at time: 0 min
    - E. Coli 2139-06154 NDM5 with Meropenem - 8ug/ml at time: 45 min
    - E. Coli 2139-06154 NDM5 with Meropenem - 8ug/ml at time: 120 min

In [20]:
data_overview.iloc[[1]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
1,11.05.2022,Nanomotion measurement,2139-06154 NDM5 Ecoli (Resistent Mero),10 um/ R308_73 / depth 903 um,None,Resistant cells with added meropenem 8ug/ml,Variance goes up after adding meropenem (so yo...


In [21]:
ecoli_NDM5_df = pd.DataFrame(columns=['Time [s]', 'Signal','File Name', 'What', 'Strain', 'Chip','Drums'])
ecoli_NDM5_df = import_data_from_folder('2022.05.11', ecoli_NDM5_df, data_overview, index=1, naming_format = 'legacy')

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [22]:
ecoli_NDM5_CTR_df = ecoli_NDM5_df[~ecoli_NDM5_df['File Name'].str.contains('meropenem')] # the ~ is to exclude the cases with the string 'meropenem'
ecoli_NDM5_WA_45min_df = ecoli_NDM5_df[ecoli_NDM5_df['File Name'].str.contains('45min')]
ecoli_NDM5_WA_120min_df = ecoli_NDM5_df[ecoli_NDM5_df['File Name'].str.contains('120min')]
ecoli_NDM5_WA_0min_df = ecoli_NDM5_df[~ecoli_NDM5_df['File Name'].str.contains('45min|120min')] #Intermediate step to have cases without antibiotic and with antibiotic but at time 0.
ecoli_NDM5_WA_0min_df = ecoli_NDM5_WA_0min_df[ecoli_NDM5_WA_0min_df['File Name'].str.contains('meropenem')] #final step to have only cases with antibiotic with time 0. 

In [23]:
#Deleting the file 'E_coli_NDM5_R308_72_11_05_22_0.txt' from the dataframe, as it only has 0.9 seconds of length.
ecoli_NDM5_CTR_df = ecoli_NDM5_CTR_df.drop(ecoli_NDM5_CTR_df[(ecoli_NDM5_CTR_df['Time [s]'] != 29.9995)].index)

In [24]:
ecoli_NDM5_CTR = dataframe_to_arrays(dataframe_column = ecoli_NDM5_CTR_df['Signal'])
ecoli_NDM5_WA_0min = dataframe_to_arrays(dataframe_column = ecoli_NDM5_WA_0min_df['Signal'])
ecoli_NDM5_WA_45min = dataframe_to_arrays(dataframe_column = ecoli_NDM5_WA_45min_df['Signal'])
ecoli_NDM5_WA_120min = dataframe_to_arrays(dataframe_column = ecoli_NDM5_WA_120min_df['Signal'])

### Import folder '2022.05.13' with E. Coli 881-61610 blAOXA-48 (Sensitive Mero) (Case 2)

Import of Folder '2022.05.13' which corresponds of E. Coli 881-61610 blAOXA-48 (Sensitive Mero) with and without the antibiotic Meropenem with a concentration of 8ug/ml.

Cases inside this folder: 

    - E. Coli 881-61610 blAOXA-48 Control
    - E. Coli 881-61610 blAOXA-48 Meropenem - 8ug/ml at time: 0 min
    - E. Coli 881-61610 blAOXA-48 Meropenem - 8ug/ml at time: 60 min
    - E. Coli 881-61610 blAOXA-48 with Meropenem - 8ug/ml at time: 120 min

In [25]:
data_overview.iloc[[2]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
2,13.05.2022,Nanomotion measurement,881-61610 blAOXA-48 (Sensitive Mero),R308_73/ 10 um/ depth 903 um,None,Sensitive cells with added meropenem10ug/ml. m...,Variance after addition of meropenem went to a...


In [26]:
ecoli_blAOXA48_df = pd.DataFrame(columns=['Time [s]', 'Signal','File Name', 'What', 'Strain', 'Chip','Drums'])
ecoli_blAOXA48_df = import_data_from_folder('2022.05.13', ecoli_blAOXA48_df, data_overview, index=2, naming_format = 'legacy')

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [27]:
ecoli_blAOXA48_CTR_df = ecoli_blAOXA48_df[~ecoli_blAOXA48_df['File Name'].str.contains('meropenem')] # the ~ is to exclude the cases with the string 'meropenem'
ecoli_blAOXA48_WA_60min_df = ecoli_blAOXA48_df[ecoli_blAOXA48_df['File Name'].str.contains('60min')]
ecoli_blAOXA48_WA_120min_df = ecoli_blAOXA48_df[ecoli_blAOXA48_df['File Name'].str.contains('120min')]
ecoli_blAOXA48_WA_0min_df = ecoli_blAOXA48_df[~ecoli_blAOXA48_df['File Name'].str.contains('60min|120min')] #Intermediate step to have cases without antibiotic and with antibiotic but at time 0.
ecoli_blAOXA48_WA_0min_df = ecoli_blAOXA48_WA_0min_df[ecoli_blAOXA48_WA_0min_df['File Name'].str.contains('meropenem')] #final step to have only cases with antibiotic with time 0. 

In [28]:
ecoli_blAOXA48_CTR = dataframe_to_arrays(dataframe_column = ecoli_blAOXA48_CTR_df['Signal'])
ecoli_blAOXA48_WA_60min = dataframe_to_arrays(dataframe_column = ecoli_blAOXA48_WA_60min_df['Signal'])
ecoli_blAOXA48_WA_120min = dataframe_to_arrays(dataframe_column = ecoli_blAOXA48_WA_120min_df['Signal'])
ecoli_blAOXA48_WA_0min = dataframe_to_arrays(dataframe_column = ecoli_blAOXA48_WA_0min_df['Signal'])

### Import folder '2022.05.17' with E. Coli 880-92227 blaOXA-244 (Sensitive Mero) (Case 3)

Import of Folder '2022.05.17' which corresponds of E. Coli 880-92227 blaOXA-244 (Sensitive Mero) with and without the antibiotic Meropenem with a concentration of 8ug/ml.

Cases inside this folder: 

    - E. Coli 880-92227 blaOXA-244 Control
    - E. Coli 880-92227 blaOXA-244 Meropenem - 8ug/ml at time: 0 min
    - E. Coli 880-92227 blaOXA-244 with Meropenem - 8ug/ml at time: 120 min

In [29]:
data_overview.iloc[[3]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
3,17.05.2022,Nanomotion measurement,880-92227 blaOXA-244 (Sensitive Mero),R308_76/ 10 um/ depth 1258 um,None,None,None


In [30]:
ecoli_blaOXA244_df = pd.DataFrame(columns=['Time [s]', 'Signal','File Name', 'What', 'Strain', 'Chip','Drums'])
ecoli_blaOXA244_df = import_data_from_folder('2022.05.17', ecoli_blaOXA244_df, data_overview, index=3, naming_format = 'legacy')

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [31]:
ecoli_blaOXA244_CTR_df = ecoli_blaOXA244_df[~ecoli_blaOXA244_df['File Name'].str.contains('meropenem')] # the ~ is to exclude the cases with the string 'meropenem'
ecoli_blaOXA244_WA_120min_df = ecoli_blaOXA244_df[ecoli_blaOXA244_df['File Name'].str.contains('120min')]
ecoli_blaOXA244_WA_0min_df = ecoli_blaOXA244_df[~ecoli_blaOXA244_df['File Name'].str.contains('run2|120min')] #Intermediate step to have cases without antibiotic and with antibiotic but at time 0.
ecoli_blaOXA244_WA_0min_df = ecoli_blaOXA244_WA_0min_df[ecoli_blaOXA244_WA_0min_df['File Name'].str.contains('meropenem')] #final step to have only cases with antibiotic with time 0. 

In [32]:
ecoli_blaOXA244_CTR = dataframe_to_arrays(dataframe_column = ecoli_blaOXA244_CTR_df['Signal'])
ecoli_blaOXA244_WA_120min = dataframe_to_arrays(dataframe_column = ecoli_blaOXA244_WA_120min_df['Signal'])
ecoli_blaOXA244_WA_0min = dataframe_to_arrays(dataframe_column = ecoli_blaOXA244_WA_0min_df['Signal'])

### Import folder '2022.05.18' with E. Coli 1303-49082-01 NDM7 (Resistent Mero) (Case 4)

Import of Folder '2022.05.18' which corresponds of E. Coli 1303-49082-01 NDM7 (Resistent Mero) with and without the antibiotic Meropenem with a concentration of 8ug/ml.

Cases inside this folder: 

    - E. Coli 1303-49082-01 NDM7 Control
    - E. Coli 1303-49082-01 NDM7 Meropenem - 8ug/ml at time: 0 min
    - E. Coli 1303-49082-01 NDM7 with Meropenem - 8ug/ml at time: 90 min

In [33]:
data_overview.iloc[[4]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
4,18.05.2022,Nanomotion measurement,1303-49082-01 NDM7 (Resistent Mero),None,None,"Resistant cells with added meropenem 8ug/ml , ...",First measurement a lot of signal then after a...


In [34]:
ecoli_NDM7_df = pd.DataFrame(columns=['Time [s]', 'Signal','File Name', 'What', 'Strain', 'Chip','Drums'])
ecoli_NDM7_df = import_data_from_folder('2022.05.18', ecoli_NDM7_df, data_overview, index=4, naming_format = 'legacy')

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [35]:
ecoli_NDM7_CTR_df = ecoli_NDM7_df[~ecoli_NDM7_df['File Name'].str.contains('Meropenem')] # the ~ is to exclude the cases with the string 'meropenem'
ecoli_NDM7_WA_90min_df = ecoli_NDM7_df[ecoli_NDM7_df['File Name'].str.contains('90min')]
ecoli_NDM7_WA_0min_df = ecoli_NDM7_df[~ecoli_NDM7_df['File Name'].str.contains('run|90min')] #Intermediate step to have cases without antibiotic and with antibiotic but at time 0.
ecoli_NDM7_WA_0min_df = ecoli_NDM7_WA_0min_df[ecoli_NDM7_WA_0min_df['File Name'].str.contains('Meropenem')] #final step to have only cases with antibiotic with time 0. 

In [36]:
ecoli_NDM7_CTR = dataframe_to_arrays(dataframe_column = ecoli_NDM7_CTR_df['Signal'])
ecoli_NDM7_WA_90min = dataframe_to_arrays(dataframe_column = ecoli_NDM7_WA_90min_df['Signal'])
ecoli_NDM7_WA_0min = dataframe_to_arrays(dataframe_column = ecoli_NDM7_WA_0min_df['Signal'])

### Import folder '2022.05.19' with E. Coli 1941-17928 NDM7 (Resistent Mero) (Case 5)

Import of Folder '2022.05.19' which corresponds of E. Coli 1941-17928 NDM7 (Resistent Mero) with and without the antibiotic Meropenem with a concentration of 8ug/ml.

Cases inside this folder: 

    - E. Coli 1941-17928 NDM7 Control
    - E. Coli 1941-17928 NDM7 with Meropenem - 8ug/ml at time: 0 min
    - E. Coli 1941-17928 NDM7 with Meropenem - 8ug/ml at time: 60 min
    - E. Coli 1941-17928 NDM7 with Meropenem - 8ug/ml at time: 120 min

In [37]:
data_overview.iloc[[5]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
5,19.05.2022,Nanomotion measurement,1941-17928 NDM7 (Resistent Mero),R308_77/ 8 um/ depth 1258 um,None,The first measurement without the drug showed ...,"First measurement variance of 2000, this is no..."


In [38]:
ecoli_NDM7_2_df = pd.DataFrame(columns=['Time [s]', 'Signal','File Name', 'What', 'Strain', 'Chip','Drums'])
ecoli_NDM7_2_df = import_data_from_folder('2022.05.19', ecoli_NDM7_2_df, data_overview, index=5, naming_format = 'legacy')

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [39]:
ecoli_NDM7_2_CTR_df = ecoli_NDM7_2_df[~ecoli_NDM7_2_df['File Name'].str.contains('meropenem')] # the ~ is to exclude the cases with the string 'meropenem'
ecoli_NDM7_2_WA_60min_df = ecoli_NDM7_2_df[ecoli_NDM7_2_df['File Name'].str.contains('60min')]
ecoli_NDM7_2_WA_120min_df = ecoli_NDM7_2_df[ecoli_NDM7_2_df['File Name'].str.contains('120min')]
ecoli_NDM7_2_WA_0min_df = ecoli_NDM7_2_df[~ecoli_NDM7_2_df['File Name'].str.contains('60min|120min')] #Intermediate step to have cases without antibiotic and with antibiotic but at time 0.
ecoli_NDM7_2_WA_0min_df = ecoli_NDM7_2_WA_0min_df[ecoli_NDM7_2_WA_0min_df['File Name'].str.contains('meropenem')] #final step to have only cases with antibiotic with time 0. 

In [40]:
ecoli_NDM7_2_CTR = dataframe_to_arrays(dataframe_column = ecoli_NDM7_2_CTR_df['Signal'])
ecoli_NDM7_2_WA_60min = dataframe_to_arrays(dataframe_column = ecoli_NDM7_2_WA_60min_df['Signal'])
ecoli_NDM7_2_WA_120min = dataframe_to_arrays(dataframe_column = ecoli_NDM7_2_WA_120min_df['Signal'])
ecoli_NDM7_2_WA_0min = dataframe_to_arrays(dataframe_column = ecoli_NDM7_2_WA_0min_df['Signal'])

### Import folder '2022.05.20' with E. Coli 1544-15672-03 blaOXA-48 (Sensitive Mero) (Case 6)

Import of Folder '2022.05.20' which corresponds of E. Coli 1544-15672-03 blaOXA-48 (Sensitive Mero) with and without the antibiotic Meropenem with a concentration of 8ug/ml.

Cases inside this folder: 

    - E. Coli 1544-15672-03 blaOXA-48 Control
    - E. Coli 1544-15672-03 blaOXA-48 with Meropenem - 8ug/ml at time: 0 min
    - E. Coli 1544-15672-03 blaOXA-48 with Meropenem - 8ug/ml at time: 60 min
    - E. Coli 1544-15672-03 blaOXA-48 with Meropenem - 8ug/ml at time: 120 min

In [41]:
data_overview.iloc[[6]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
6,20.05.2022,Nanomotion measurement,1544-15672-03 bla OXA-48 like (Sensitive Mero),R308_75/ 10 um/ depth 1258 um,100 metingen 30 s,None,Variance voor antibioticum 8 daarna naar achte...


In [42]:
ecoli_blaOXA48_2_df = pd.DataFrame(columns=['Time [s]', 'Signal','File Name', 'What', 'Strain', 'Chip','Drums'])
ecoli_blaOXA48_2_df = import_data_from_folder('2022.05.20', ecoli_blaOXA48_2_df, data_overview, index=6, naming_format = 'legacy')

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [43]:
ecoli_blaOXA48_2_CTR_df = ecoli_blaOXA48_2_df[~ecoli_blaOXA48_2_df['File Name'].str.contains('Mero')] # the ~ is to exclude the cases with the string 'meropenem'
ecoli_blaOXA48_2_WA_60min_df = ecoli_blaOXA48_2_df[ecoli_blaOXA48_2_df['File Name'].str.contains('60min')]
ecoli_blaOXA48_2_WA_120min_df = ecoli_blaOXA48_2_df[ecoli_blaOXA48_2_df['File Name'].str.contains('120min')]
ecoli_blaOXA48_2_WA_0min_df = ecoli_blaOXA48_2_df[~ecoli_blaOXA48_2_df['File Name'].str.contains('60min|120min')] #Intermediate step to have cases without antibiotic and with antibiotic but at time 0.
ecoli_blaOXA48_2_WA_0min_df = ecoli_blaOXA48_2_WA_0min_df[ecoli_blaOXA48_2_WA_0min_df['File Name'].str.contains('Mero')] #final step to have only cases with antibiotic with time 0.

In [44]:
ecoli_blaOXA48_2_CTR = dataframe_to_arrays(dataframe_column = ecoli_blaOXA48_2_CTR_df['Signal'])
ecoli_blaOXA48_2_WA_60min = dataframe_to_arrays(dataframe_column = ecoli_blaOXA48_2_WA_60min_df['Signal'])
ecoli_blaOXA48_2_WA_120min = dataframe_to_arrays(dataframe_column = ecoli_blaOXA48_2_WA_120min_df['Signal'])
ecoli_blaOXA48_2_WA_0min = dataframe_to_arrays(dataframe_column = ecoli_blaOXA48_2_WA_0min_df['Signal'])

### Import folder '2022.12.08 - with E. Coli' with E. Coli 1303 bacteria (Sensitive or resistant Mero) (Case 15)

Import of Folder '2022.12.08 - with E. Coli' which corresponds to E. Coli 1303 bacteria (Sensitive or resistant Mero) with and without the antibiotic Meropenem with a concentration of 10ug/ml.

Annonations: Empty Drums are in the folder "2022.12.08 - MH Graphene control".

Cases inside this folder: 

    - E. Coli 1303 Control No Meropenem - 0ug/ml at time: 0 min
    - E. Coli 1303 with Meropenem - 10ug/ml at time: 30 min

In [45]:
data_overview.iloc[[15]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
15,08.12.2022,Nanomotion measurement (with graphene),E.coli 1303,R308_93,8um,"from MH agar to MH broth, imi 10ug(30 min), OD...",None


In [46]:
ecoli_1303_df = pd.DataFrame(columns=['Time [s]', 'Signal', 'Chip Name', 'Species','Strain', 'Antibiotic', 'Concentration [\u03BCg]', 'Time [min]', 'Drum'])
ecoli_1303_df = import_data_from_folder('2022.12.08 - with E. Coli', ecoli_1303_df, data_overview, index = 15, naming_format = 'new_naming') 

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [47]:
ecoli_1303_CTR_df = ecoli_1303_df[(ecoli_1303_df['Species'].str.contains('Ecoli')) & 
                             (ecoli_1303_df['Antibiotic'].str.contains('Noab'))]
# ecoli_1303_WA_30min_df = ecoli_1303_df[ecoli_1303_df['Time [min]'].str.contains('30')]

In [48]:
ecoli_1303_CTR = dataframe_to_arrays(dataframe_column = ecoli_1303_CTR_df['Signal']) 
# ecoli_1303_WA_30min = dataframe_to_arrays(dataframe_column = ecoli_1303_WA_30min_df['Signal'])

### Import folder '2023.05.25' with E. Coli 1544 bacteria (Sensitive or resistant Mero) (Case 41)

Import of Folder '2023.05.25' which corresponds to E. Coli 1544 bacteria (Sensitive or resistant Mero) with and without the antibiotic Meropenem with a concentration of 1 ug/ml.

Annonations: N/A.

Cases inside this folder: 

    - Empty Drums
    - E. Coli 1544 Control No Meropenem - 0ug/ml at time: 0 min
    - E. Coli 1544 with Meropenem - 1 ug/ml at time: 60 min

In [49]:
data_overview.iloc[[41]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
41,2023.25.05,Nanomotion measurement (with graphene),E.coli 1544,R308_121,8um,"MH broth, OD600=0,18, mero 1ug/ml, 100 metinge...",None


In [50]:
ecoli_1544_df = pd.DataFrame(columns=['Time [s]', 'Signal', 'Chip Name', 'Species','Strain', 'Antibiotic', 'Concentration [\u03BCg]', 'Time [min]', 'Drum'])
ecoli_1544_df = import_data_from_folder('2023.05.25', ecoli_1544_df, data_overview, index = 41, naming_format = 'new_naming') 

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [51]:
ecoli_1544_empty_drums_df = ecoli_1544_df[ecoli_1544_df['Species'].str.contains('Mhcontrol')]
ecoli_1544_CTR_df = ecoli_1544_df[(ecoli_1544_df['Species'].str.contains('Ecoli')) & 
                             (ecoli_1544_df['Antibiotic'].str.contains('Noab'))]
# ecoli_1544_WA_60min_df = ecoli_1544_df[ecoli_1544_df['Time [min]'].str.contains('60')]

In [52]:
ecoli_1544_empty_drums = dataframe_to_arrays(dataframe_column = ecoli_1544_empty_drums_df['Signal'])
ecoli_1544_CTR = dataframe_to_arrays(dataframe_column = ecoli_1544_CTR_df['Signal'])
# ecoli_1544_WA_60min = dataframe_to_arrays(dataframe_column = ecoli_1544_WA_60min_df['Signal'])

### Import folder '2023.05.31' with E. Coli ATCC 25922 bacteria (Sensitive or resistant Mero) (Case 41)

Import of Folder '2023.05.25' which corresponds to E. Coli ATCC 25922 bacteria (Sensitive or resistant Mero) with and without the antibiotic Meropenem with a concentration of 2 ug/ml.

Annonations: N/A.

Cases inside this folder: 

    - Empty Drums
    - E. Coli ATCC 25922 Control No Meropenem - 0ug/ml at time: 0 min
    - E. Coli ATCC 25922 with Meropenem - 1 ug/ml at time: 15 min
    - E. Coli ATCC 25922 with Meropenem - 2 ug/ml at time: 15 min
    - E. Coli ATCC 25922 with Meropenem - 0.5 ug/ml at time: 30 min

In [53]:
data_overview.iloc[[42, 43]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
42,2023.31.05,Nanomotion measurement (with graphene),E.coli ATCC 25922,R308_124,8um,"MH broth | OD600=0,19 | mero 1-3ug/ml, incubat...","Tried measuring a gradient, did not work as ex..."
43,2023.31.05,Nanomotion measurement (with graphene),E.coli ATCC 25922,R308_125,8um,"MH broth | OD600=0,21 | mero 0,5-2ug/ml, incub...",None


In [54]:
ecoli_25922_df = pd.DataFrame(columns=['Time [s]', 'Signal', 'Chip Name', 'Species','Strain', 'Antibiotic', 'Concentration [\u03BCg]', 'Time [min]', 'Drum'])
ecoli_25922_df = import_data_from_folder('2023.05.31', ecoli_25922_df, data_overview, index = 42, naming_format = 'new_naming') 

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [55]:
ecoli_25922_empty_drums_df = ecoli_25922_df[ecoli_25922_df['Species'].str.contains('Mh-control')]
ecoli_25922_CTR_df = ecoli_25922_df[(ecoli_25922_df['Species'].str.contains('Ecoli')) & 
                             (ecoli_25922_df['Antibiotic'].str.contains('Noab'))]
# ecoli_25922_WA_15min_df = ecoli_25922_df[ecoli_25922_df['Time [min]'].str.contains('15')]
# ecoli_25922_WA_30min_df = ecoli_25922_df[ecoli_25922_df['Time [min]'].str.contains('30')]

In [56]:
ecoli_25922_empty_drums = dataframe_to_arrays(dataframe_column = ecoli_25922_empty_drums_df['Signal'])
ecoli_25922_CTR = dataframe_to_arrays(dataframe_column = ecoli_25922_CTR_df['Signal'])
# ecoli_25922_WA_15min = dataframe_to_arrays(dataframe_column = ecoli_25922_WA_15min_df['Signal'])
# ecoli_25922_WA_30min = dataframe_to_arrays(dataframe_column = ecoli_25922_WA_30min_df['Signal'])

# Import Pseudomonas aeruginosa Bacteria

### Import folder '2023.03.01' with Pseudomonas aeruginosa L1171 (Sensitive Mero) (Case 26)

Import of Folder '2023.03.01' which corresponds of Pseudomonas aeruginosa L1171 (Sensitive Mero) with and without the antibiotic Meropenem with a concentration of 50ug/ml. Empty Drums case are also present in this folder.

Cases inside this folder: 

    - Empty Drums cases
    - Paeruginosa L1171 Control No Meropenem - 0ug/ml at time: 0 min
    - Paeruginosa L1171 with Meropenem - 50ug/ml at time: 60 min
    - Paeruginosa L1171 with Meropenem - 50ug/ml at time: 120 min

In [57]:
data_overview.iloc[[26]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
26,01.03.2023,Nanomotion measurement (with graphene),P.aeruginosa L1171,R308_97,8um,"MH broth, OD600=0,31, mero 50ug/ml, 95 metinge...",Good outcome. Signal dropped significantly aft...


In [58]:
Paeruginosa_df = pd.DataFrame(columns=['Time [s]', 'Signal', 'Chip Name', 'Species','Strain', 'Antibiotic', 'Concentration [\u03BCg]', 'Time [min]', 'Drum'])
Paeruginosa_df = import_data_from_folder('2023.03.01', Paeruginosa_df, data_overview, index = 26, naming_format = 'new_naming') 

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [59]:
Paeruginosa_empty_drums_df = Paeruginosa_df[Paeruginosa_df['Species'].str.contains('Nocells')]
Paeruginosa_L1171_CTR_df = Paeruginosa_df[(Paeruginosa_df['Species'].str.contains('Paeruginosa')) & 
                             (Paeruginosa_df['Antibiotic'].str.contains('Noab'))]
# Paeruginosa_L1171_WA_60min_df = Paeruginosa_df[Paeruginosa_df['Time [min]'].str.contains('60')]
# Paeruginosa_L1171_WA_120min_df = Paeruginosa_df[Paeruginosa_df['Time [min]'].str.contains('120')]

In [60]:
paeruginosa_empty_drums = dataframe_to_arrays(dataframe_column = Paeruginosa_empty_drums_df['Signal'])
Paeruginosa_L1171_CTR = dataframe_to_arrays(dataframe_column = Paeruginosa_L1171_CTR_df['Signal'])
# Paeruginosa_L1171_WA_60min = dataframe_to_arrays(dataframe_column = Paeruginosa_L1171_WA_60min_df['Signal'])
# Paeruginosa_L1171_WA_120min = dataframe_to_arrays(dataframe_column = Paeruginosa_L1171_WA_120min_df['Signal'])

# Import of Acinetobacter Baumannii Bacteria

### Import folder '2023.03.07' with Acinetobacter baumannii B1380 (Sensitive or resistant Mero) (Case 30)

Import of Folder '2023.03.07' which corresponds to Acinetobacter baumannii B1380 (Sensitive Mero) with and without the antibiotic Ciprofloxacin with a concentration of 50ug/ml. Empty Drums case are also present in this folder.

Annonations: 'Looks like multiple bacteria on camera'

Cases inside this folder: 

    - Empty Drums cases
    - Acinetobacter baumannii B1380 Control No Ciprofloxacin - 0ug/ml at time: 0 min
    - Acinetobacter baumannii B1380 with Ciprofloxacin - 50ug/ml at time: 60 min
    - Acinetobacter baumannii B1380 with Ciprofloxacin - 50ug/ml at time: 150 min

In [61]:
data_overview.iloc[[27, 30]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
27,02.03.2023,Nanomotion measurement (with graphene),A.baumanii B1380,R308_98,8um,"MH broth, OD600=0,24, cipro 50ug/ml, 125 metin...","signal is very scattered, not sure if resuslts..."
30,07.03.2023,Nanomotion measurement (with graphene),A.baumanii B1380,R308_99,8um,"MH broth, OD600=0,36, cipro 50ug/ml, 106 metin...",Looks like multiple bacteria on camera


In [62]:
abaumannii_B1380_df = pd.DataFrame(columns=['Time [s]', 'Signal', 'Chip Name', 'Species','Strain', 'Antibiotic', 'Concentration [\u03BCg]', 'Time [min]', 'Drum'])
abaumannii_B1380_df = import_data_from_folder('2023.03.07', abaumannii_B1380_df, data_overview, index = 30, naming_format = 'new_naming') 

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [63]:
abaumannii_B1380_empty_drums_df = abaumannii_B1380_df[abaumannii_B1380_df['Species'].str.contains('Nospecies')]
abaumannii_B1380_CTR_df = abaumannii_B1380_df[(abaumannii_B1380_df['Species'].str.contains('Abaumannii')) & 
                             (abaumannii_B1380_df['Antibiotic'].str.contains('Noab'))]
# abaumannii_B1380_WA_60min_df = abaumannii_B1380_df[abaumannii_B1380_df['Time [min]'].str.contains('60')]
# abaumannii_B1380_WA_150min_df = abaumannii_B1380_df[abaumannii_B1380_df['Time [min]'].str.contains('150')]

In [64]:
abaumannii_B1380_empty_drums = dataframe_to_arrays(dataframe_column = abaumannii_B1380_empty_drums_df['Signal'])
abaumannii_B1380_CTR = dataframe_to_arrays(dataframe_column = abaumannii_B1380_CTR_df['Signal'])
# abaumannii_B1380_WA_60min = dataframe_to_arrays(dataframe_column = abaumannii_B1380_WA_60min_df['Signal'])
# abaumannii_B1380_WA_150min = dataframe_to_arrays(dataframe_column = abaumannii_B1380_WA_150min_df['Signal'])

# Import Staphylococcus Aureus Bacteria

### Import folder '2023.03.14' with Staphylococcus Aureus L4443 (Case 32)

Import of Folder '2023.03.14' which corresponds to Staphylococcus Aureus L4443 with and without the antibiotic Ciprofloxacin with a concentration of 50ug/ml.

Annonations: 'The measurements with antibiotics are not considered, as in the notes of this experiment says that the camera was moving during while measuring with ab.'

Cases inside this folder: 

    - Staphylococcus Aureus L4443 Control No Ciprofloxacin - 0ug/ml at time: 0 min
    - Staphylococcus Aureus L4443 with Ciprofloxacin - 50ug/ml at time: 60 min

### Note: The folder '2023.09.02' was not imported as the brpth was contaminated and the experiment was postponed. 

In [65]:
data_overview.iloc[[32]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
32,14.03.2023,Nanomotion measurement (with graphene),S.aureus L4443,R308_101,8um,"MH broth, OD600=0,31, cipro 50ug/ml, 103 metin...","Measurements after adding ab are invalid, came..."


In [66]:
S_aureus_L4443_df = pd.DataFrame(columns=['Time [s]', 'Signal', 'Chip Name', 'Species','Strain', 'Antibiotic', 'Concentration [\u03BCg]', 'Time [min]', 'Drum'])
S_aureus_L4443_df = import_data_from_folder('2023.03.14', S_aureus_L4443_df, data_overview, index = 32, naming_format = 'new_naming') 

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [67]:
S_aureus_L4443_CTR_df = S_aureus_L4443_df[(S_aureus_L4443_df['Species'].str.contains('Sareus')) & 
                             (S_aureus_L4443_df['Antibiotic'].str.contains('Noab'))]
# S_aureus_L4443_WA_60min_df = S_aureus_L4443_df[S_aureus_L4443_df['Time [min]'].str.contains('60')]

In [68]:
S_aureus_L4443_CTR = dataframe_to_arrays(dataframe_column = S_aureus_L4443_CTR_df['Signal'])
# S_aureus_L4443_WA_60min = dataframe_to_arrays(dataframe_column = S_aureus_L4443_WA_60min_df['Signal'])

### Import folder '2023.03.21' with Staphylococcus Aureus L4565 (Case 35)

Import of Folder '2023.03.21' which corresponds to Staphylococcus Aureus L4565 with and without the antibiotic Ciprofloxacin with a concentration of 60ug/ml.

Annonations: 'The measurements of empty chips are not good due to driffting'

Cases inside this folder: 

    - Staphylococcus Aureus L4565 Control No Ciprofloxacin - 0ug/ml at time: 0 min
    - Staphylococcus Aureus L4565 with Ciprofloxacin - 60ug/ml at time: 60 min
    - Staphylococcus Aureus L4565 with Ciprofloxacin - 60ug/ml at time: 120 min

In [69]:
data_overview.iloc[[35]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
35,21.03.2023,Nanomotion measurement (with graphene),S.aureus MRSA L4565 (Resistant),R308_105,8um,"MH broth, OD600=0,31, amoxi 60ug/ml, 101 metin...",Measurements of empty chip are not good (drift...


In [70]:
S_aureus_L4565_df = pd.DataFrame(columns=['Time [s]', 'Signal', 'Chip Name', 'Species','Strain', 'Antibiotic', 'Concentration [\u03BCg]', 'Time [min]', 'Drum'])
S_aureus_L4565_df = import_data_from_folder('2023.03.21', S_aureus_L4565_df, data_overview, index = 32, naming_format = 'new_naming') 

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [71]:
S_aureus_L4565_CTR_df = S_aureus_L4565_df[(S_aureus_L4565_df['Species'].str.contains('Mrsa')) & 
                             (S_aureus_L4565_df['Antibiotic'].str.contains('Noab'))]
# S_aureus_L4565_WA_60min_df = S_aureus_L4565_df[S_aureus_L4565_df['Time [min]'].str.contains('60')]
# S_aureus_L4565_WA_120min_df = S_aureus_L4565_df[S_aureus_L4565_df['Time [min]'].str.contains('120')]

In [72]:
S_aureus_L4565_CTR = dataframe_to_arrays(dataframe_column = S_aureus_L4565_CTR_df['Signal'])
# S_aureus_L4565_WA_60min = dataframe_to_arrays(dataframe_column = S_aureus_L4565_WA_60min_df['Signal'])
# S_aureus_L4565_WA_120min = dataframe_to_arrays(dataframe_column = S_aureus_L4565_WA_120min_df['Signal'])

### Merge of all Staphylococcus Aureus cases:

In [73]:
S_aureus_CTR = np.vstack((S_aureus_L4443_CTR, S_aureus_L4565_CTR))
# S_aureus_WA_60min = np.vstack((S_aureus_L4443_WA_60min, S_aureus_L4565_WA_60min))
# S_aureus_WA_120min = np.vstack((S_aureus_L4565_WA_120min))

# Import Salmonella Enteritidis Bacteria

### Import folder '2023.03.16' and '2023.03.20' with Salmonella Enteritidis S1400 (Case 34)

Import of Folder '2023.03.16' and '2023.03.20' which corresponds to Salmonella Enteritidis S1400 (Sensitive Mero) with and without the antibiotic Ciprofloxacin with a concentration of 50ug/ml. Empty Drums case are also present in this folder.

Annonations: 'Looks like multiple bacteria on camera'

Cases inside this folder: 

    - Empty Drums cases
    - Salmonella Enteritidis S1400 Control No Ciprofloxacin - 0ug/ml at time: 0 min
    - Salmonella Enteritidis S1400 with Ciprofloxacin - 50ug/ml at time: 60 min
    - Salmonella Enteritidis S1400 with Ciprofloxacin - 50ug/ml at time: 120 min

In [74]:
data_overview.loc[[33, 34]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
33,16.03.2023,Nanomotion measurement (with graphene),S.enteriditis S1400,R308_102,8um,"MH broth, OD600=0,20, cipro 50ug/ml, 125 metin...","low signal, flushed chamber with MH after lett..."
34,20.03.2023,Nanomotion measurement (with graphene),S.enteriditis S1400,R308_104,8um,"MH broth, OD600=0,24, cipro 50ug/ml, 100 metin...","Low signal, significant drop in the signal aft..."


In [75]:
Senteriditis_S1400_df = pd.DataFrame(columns=['Time [s]', 'Signal', 'Chip Name', 'Species','Strain', 'Antibiotic', 'Concentration [\u03BCg]', 'Time [min]', 'Drum'])
Senteriditis_S1400_df = import_data_from_folder('2023.03.16', Senteriditis_S1400_df, data_overview, index = 33, naming_format = 'new_naming') 
Senteriditis_S1400_df = import_data_from_folder('2023.03.20', Senteriditis_S1400_df, data_overview, index = 34, naming_format = 'new_naming') 

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [76]:
Senteriditis_S1400_empty_drums_df = Senteriditis_S1400_df[Senteriditis_S1400_df['Species'].str.contains('Nospecies')]
Senteriditis_S1400_CTR_df = Senteriditis_S1400_df[(Senteriditis_S1400_df['Species'].str.contains('Senteritidis')) & 
                             (Senteriditis_S1400_df['Antibiotic'].str.contains('Noab'))]
Senteriditis_S1400_CTR_2_df = Senteriditis_S1400_df[(Senteriditis_S1400_df['Species'].str.contains('Sentertidis')) & 
                             (Senteriditis_S1400_df['Antibiotic'].str.contains('Noab'))] #this last one is included due to a typo in the naming of the files
# Senteriditis_S1400_WA_60min_df = Senteriditis_S1400_df[Senteriditis_S1400_df['Time [min]'].str.contains('60')]
# Senteriditis_S1400_WA_120min_df = Senteriditis_S1400_df[Senteriditis_S1400_df['Time [min]'].str.contains('120')]

In [77]:
senteriditis_S1400_empty_drums = dataframe_to_arrays(dataframe_column = Senteriditis_S1400_empty_drums_df['Signal'])
senteriditis_S1400_CTR_1 = dataframe_to_arrays(dataframe_column = Senteriditis_S1400_CTR_df['Signal'])
senteriditis_S1400_CTR_2 = dataframe_to_arrays(dataframe_column = Senteriditis_S1400_CTR_2_df['Signal'])
# senteriditis_S1400_WA_60min = dataframe_to_arrays(dataframe_column = Senteriditis_S1400_WA_60min_df['Signal'])
# senteriditis_S1400_WA_120min = dataframe_to_arrays(dataframe_column = Senteriditis_S1400_WA_120min_df['Signal'])


### Merge of all cases Salmonella Enteritidis S1400

In [78]:
senteriditis_S1400_CTR = np.vstack((senteriditis_S1400_CTR_1, senteriditis_S1400_CTR_2))

# Import of Klebsiella Pneumoniae Bacteria

### Import folder '2023.12.06' and '2023.12.07' with Klebsiella pneumoniae KPC2 bacteria (Susceptible to Imipenem) (Case 12 and Case 14)

Import of Folder '2023.12.06' and '2023.12.07' which correspond to Klebsiella pneumoniae KPC2 bacteria (susceptible) with and without the antibiotic Imipenem with a concentration of 10ug/ml and 20ug/ml.

Annonations: None.

Cases inside folder '2023.12.06':

    - K. Pneumoniae KPC2 Control No Imipenem - 0ug/ml at time: 0 min
    - K. Pneumoniae KPC2 with Imipenem - 20ug/ml at time: 30 min

Cases inside folder '2023.12.07':

    - Empty Drums cases
    - K. Pneumoniae KPC2 Control No Imipenem - 0ug/ml at time: 0 min
    - K. Pneumoniae KPC2 with Imipenem - 10ug/ml at time: 30 min

### Import folder '2023.04.05' and '2023.04.05' with Klebsiella pneumoniae KPC2 bacteria (Resistent to Amoxi) (Case 37 and Case 40)

Import of Folder '2023.04.05' and '2023.04.05' which correspond to Klebsiella pneumoniae KPC2 bacteria (Resistant) with and without the antibiotic Amoxicillin with a concentration of 60ug/ml.

Annonations: None.

Cases inside folder '2023.04.05':

    - Empty Drums cases
    - K. Pneumoniae KPC2 Control No Amoxicillin - 0ug/ml at time: 0 min
    - K. Pneumoniae KPC2 with Amoxicillin - 60ug/ml at time: 60 min
    - K. Pneumoniae KPC2 with Amoxicillin - 60ug/ml at time: 120 min

Cases inside folder '2023.05.04':

    - Empty Drums cases
    - K. Pneumoniae ATCC700603 Control No Amoxicillin - 0ug/ml at time: 0 min
    - K. Pneumoniae ATCC700603 with Amoxicillin - 60ug/ml at time: 60 min
    - K. Pneumoniae ATCC700603 with Amoxicillin - 60ug/ml at time: 120 min

In [79]:
data_overview.iloc[[12, 14, 37, 40]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
12,06.12.2022,Nanomotion measurement (with graphene),K.pneumoniae KPC2,R308_91,8um,"from MH agar to MH broth, imi 20ug(30 min), OD...",None
14,07.12.2022,Nanomotion measurement (with graphene),K.pneumoniae KPC2,R308_92,8um,"from MH agar to MH broth, imi 10ug(30 min), OD...",None
37,05.04.2023,Nanomotion measurement (with graphene),K.pneumoniae KPC2 (Resistant),R308_112,8um,"MH broth, OD600=0,18, amoxi 60ug/ml, 100 metin...",None
40,2023.04.05,Nanomotion measurement (with graphene),K.pneumoniae ATCC 700603 (resistant),R308_114,8um,"MH broth, OD600=0,43, amoxi 60ug/ml, 100 metin...","Nice signal, signal went up as expected."


In [80]:
Kpneumoniae_df = pd.DataFrame(columns=['Time [s]', 'Signal', 'Chip Name', 'Species','Strain', 'Antibiotic', 'Concentration [\u03BCg]', 'Time [min]', 'Drum'])
Kpneumoniae_df = import_data_from_folder('2023.04.05', Kpneumoniae_df, data_overview, index = 37, naming_format = 'new_naming') #100 
Kpneumoniae_df = import_data_from_folder('2023.12.06-MH k.pneumoniae KPC2', Kpneumoniae_df, data_overview, index = 12, naming_format = 'new_naming') #200
Kpneumoniae_df = import_data_from_folder('2023.05.04', Kpneumoniae_df, data_overview, index = 40, naming_format = 'new_naming') #100
Kpneumoniae_df = import_data_from_folder('2022.12.07', Kpneumoniae_df, data_overview, index = 14, naming_format = 'new_naming') #126

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [81]:
kpneumoniae_KPC2_empty_drums_df = Kpneumoniae_df[(Kpneumoniae_df['Species'].str.contains('Mhcontrol')) | 
                                         (Kpneumoniae_df['Species'].str.contains('Nospecies'))]

kpneumoniae_KPC2_CTR_df = Kpneumoniae_df[(Kpneumoniae_df['Species'].str.contains('Kpneumoniae')) & 
                                         (Kpneumoniae_df['Antibiotic'].str.contains('Noab'))]
# kpneumoniae_KPC2_WA_susc_30min_df = Kpneumoniae_df[Kpneumoniae_df['Time [min]'].str.contains('30')]
# kpneumoniae_KPC2_WA_resis_60min_df = Kpneumoniae_df[Kpneumoniae_df['Time [min]'].str.contains('60')]
# kpneumoniae_KPC2_WA_resis_120min_df = Kpneumoniae_df[Kpneumoniae_df['Time [min]'].str.contains('120')]

In [82]:
kpneumoniae_KPC2_empty_drums = dataframe_to_arrays(dataframe_column = kpneumoniae_KPC2_empty_drums_df['Signal'])
kpneumoniae_KPC2_CTR = dataframe_to_arrays(dataframe_column = kpneumoniae_KPC2_CTR_df['Signal'])
# kpneumoniae_KPC2_WA_susc_30min = dataframe_to_arrays(dataframe_column = kpneumoniae_KPC2_WA_susc_30min_df['Signal'])
# kpneumoniae_KPC2_WA_resis_60min = dataframe_to_arrays(dataframe_column = kpneumoniae_KPC2_WA_resis_60min_df['Signal'])
# kpneumoniae_KPC2_WA_resis_120min = dataframe_to_arrays(dataframe_column = kpneumoniae_KPC2_WA_resis_120min_df['Signal'])

### Note: 
The data that we have from K. Pneumoniae is really mixed. The first two folders ('2023.12.06' and '2023.12.07') are susceptible bacteria to imipenem and they were tested only at 30 min. The other set of folders ('2023.04.05' and '2023.04.05') are resistant bacteria to amoxicilin and they were tested at 60min and 120min. 

In [83]:
kpneumoniae_KPC2_CTR.shape #200

(526, 60000)

# Import Proteus Mirabilis Bacteria

### Note: 

I see that inside folder '2023.03.23' the folder contains bacteria with the name styphi, instead of pmirabilis. I need to ask if there's a mistake there. 

In [84]:
## Insert code here

# Import Streptococcus Agalactiae Bacteria

### Import folder '2023.06.01' with Streptococcus Agalactiae ATCC 13813 bacteria (Susceptible to Penicillin) (Case 44)

Import of Folder '2023.01.06' which corresponds to Streptococcus Agalactiae ATCC 13813 bacteria (susceptible to Penicillin) with and without the antibiotic Penicillin with a concentration of 10ug/ml and 25ug/ml.

Annonations: Folders '2023.14.04' and '2023.20.04' are not being used YET, because apparently there were problems in the measurements. Almost no cells were measured. This is why only folder '2023.06.01' is being imported.

Cases inside folder '2023.06.01':

    - Empty Drums
    - S. Agalactiae ATCC 13813 Control No Penicillin - 0ug/ml at time: 0 min
    - S. Agalactiae ATCC 13813 with Penicillin - 25ug/ml at time: 60 min
    - S. Agalactiae ATCC 13813 with Penicillin - 25ug/ml at time: 120 min

In [85]:
data_overview.iloc[[38, 39, 44]]

,Datum,What,Strain,Chip,Drums,Specifics,Outcome
38,2023.12.04,Nanomotion measurement (with graphene),S.agalactiae ATCC 13813,R308_107,8um,"MH broth, OD600=0,15, peni 60ug/ml, 100 meting...",Very slow growing. Drift in MHcontrol measurem...
39,2023.20.04,Nanomotion measurement (with graphene),S.agalactiae ATCC 13813,R308_113,8um,"Nutrient Broth, OD600=0,15, peni 60ug/ml, 100 ...",There was very little growth after 4 hours of ...
44,2023.01.06,Nanomotion measurement (with graphene),S.agalactiae ATCC 13813,R308_126,8um,"MH broth | OD600=0,23 | peni 25ug/ml | 100 met...",None


In [86]:
Sagalactiae_df = pd.DataFrame(columns=['Time [s]', 'Signal', 'Chip Name', 'Species','Strain', 'Antibiotic', 'Concentration [\u03BCg]', 'Time [min]', 'Drum'])
Sagalactiae_df = import_data_from_folder('2023.06.01', Sagalactiae_df, data_overview, index = 44, naming_format = 'new_naming') #100 

/var/folders/f3/cszb_8356vjdd9gttlf5sctw0000gn/T/ipykernel_36109/689281698.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_dataframe = pd.concat([new_dataframe, pd.DataFrame(new_row)], ignore_index=True)


In [87]:
Sagalactiae_empty_drums_df = Sagalactiae_df[(Sagalactiae_df['Species'].str.contains('Mhcontrol')) | 
                                         (Sagalactiae_df['Species'].str.contains('Nospecies'))]

Sagalactiae_CTR_df = Sagalactiae_df[(Sagalactiae_df['Species'].str.contains('Sagalactiae')) & 
                                         (Sagalactiae_df['Antibiotic'].str.contains('Noab'))]
# Sagalactiae_WA_susc_60min_df = Sagalactiae_df[Sagalactiae_df['Time [min]'].str.contains('60')]
# Sagalactiae_WA_susc_120min_df = Sagalactiae_df[Sagalactiae_df['Time [min]'].str.contains('120')]

In [88]:
Sagalactiae_empty_drums = dataframe_to_arrays(dataframe_column = Sagalactiae_empty_drums_df['Signal'])
Sagalactiae_CTR = dataframe_to_arrays(dataframe_column = Sagalactiae_CTR_df['Signal'])
# Sagalactiae_WA_susc_60min = dataframe_to_arrays(dataframe_column = Sagalactiae_WA_susc_60min_df['Signal'])
# Sagalactiae_WA_susc_120min = dataframe_to_arrays(dataframe_column = Sagalactiae_WA_susc_120min_df['Signal'])

In [89]:
Sagalactiae_CTR.shape

(100, 60000)

# Merging imported cases

The imported cases are being merged toinclude all similar cases in a single array that can be used fro training and testing in a classification algorithm. The data is being merged in the following manner:


### Empty Cavities (No graphene): (828, 60000)
- empty_drums_30: (828, 60000)

### Empty Drums  (With graphene): (604, 60000)
- empty_drums_graph_1st_batch: (203, 60000)
- paeruginosa_empty_drums: (95, 60000)
- abaumannii_B1380_empty_drums: (106, 60000)
- senteriditis_S1400_empty_drums: (100, 60000)

### E. Coli control cases:  (930, 60000)

- ecoli7744_CTR: (98, 60000)
- ecoli_NDM5_CTR: (100, 60000) #Not This one any more, as it does not have graphene.
- ecoli_blAOXA48_CTR: (82, 60000)
- ecoli_blaOXA244_CTR: (200, 60000)
- ecoli_NDM7_CTR: (200, 60000)
- ecoli_NDM7_2_CTR: (100, 60000)
- ecoli_blaOXA48_2_CTR: (99, 60000)
- ecoli_1303_CTR: (149, 60000)

### E. Coli with Antibiotics (Sensitive Mero): (380, 60000)

- ecoli7744_WA: (100, 60000)
- ecoli_blAOXA48_WA_120min: (80, 60000)
- ecoli_blaOXA244_WA_120min: (100, 60000)
- ecoli_blaOXA48_2_WA_120min: (100, 60000)

### E. Coli with Antibiotics (Resistant Mero): (207, 60000)

- ecoli_NDM5_WA_120min: (100, 60000)
- ecoli_NDM7_WA_90min: (14, 60000)
- ecoli_NDM7_2_WA_120min: (93, 60000)

# Merging of cases for further analysis with Machine Learning

In [90]:
empty_drums_30.shape

(828, 60000)

## Merge of all control cases:

    - ecoli7744_CTR: Comes from Case 0, E. Coli 7740 Control
    - ecoli_NDM5_CTR: Comes from Case 1, E. Coli 2139-06154 NDM5 Control
    - ecoli_blAOXA48_CTR: Comes from Case 2, E. Coli 881-61610 blAOXA-48 Control
    - ecoli_blaOXA244_CTR: Comes from Case 3, E. Coli 880-92227 blaOXA-244 Control
    - ecoli_NDM7_CTR: Comes from Case 4, E. Coli 1303-49082-01 NDM7 Control
    - ecoli_NDM7_2_CTR: Comes from Case 5, E. Coli 1941-17928 NDM7 Control
    - ecoli_blaOXA48_2_CTR: comes from Case 6, E. Coli 1544-15672-03 blaOXA-48 Control

This case below, ecoli_CTR_old was just a test. The idea was to see the difference in variance between Contol E. Coli bacteria with and without graphene in the drum. ecoli_CTR_old has all cases with graphene and without graphene.

In [91]:
ecoli_CTR_old = np.vstack((ecoli7744_CTR, ecoli_NDM5_CTR, ecoli_blAOXA48_CTR, ecoli_blaOXA244_CTR, ecoli_NDM7_CTR, 
                        ecoli_NDM7_2_CTR, ecoli_blaOXA48_2_CTR)) #ecoli_NDM7_CTR This is with no graphene and it has 200 cases.

This case below, ecoli_CTR has all cases with E. Coli with graphene. 

In [92]:
ecoli_CTR = np.vstack((ecoli_NDM5_CTR, ecoli_blAOXA48_CTR, ecoli_blaOXA244_CTR, ecoli_NDM7_CTR, 
                       ecoli_NDM7_2_CTR, ecoli_blaOXA48_2_CTR, ecoli_1303_CTR, ecoli_1544_CTR, ecoli_25922_CTR)) 

## Merge of all Empty Drums cases:

In [93]:
empty_drums_graph = np.vstack((empty_drums_graph_1st_batch, paeruginosa_empty_drums, abaumannii_B1380_empty_drums,
                               senteriditis_S1400_empty_drums, kpneumoniae_KPC2_empty_drums))

In [94]:
empty_drums_graph.shape

(928, 60000)

##  Merge of all sensitive bacteria with antibiotic cases:

In [95]:
ecoli_WA_subs = np.vstack((ecoli7744_WA, ecoli_blAOXA48_WA_120min, ecoli_blaOXA244_WA_120min, 
                           ecoli_blaOXA48_2_WA_120min))

## Merge of all resistance bacteria with antibiotic cases

In [96]:
ecoli_WA_resis = np.vstack((ecoli_NDM5_WA_120min, ecoli_NDM7_WA_90min, ecoli_NDM7_2_WA_120min))

## Merge of all bacteria with antibiotics (both resistant and sensitive)

In [97]:
ecoli_WA_total = np.vstack((ecoli_WA_subs, ecoli_WA_resis))

## Intact Drums: Drums with graphene, both with and without bacteria

In [98]:
intact_drums = np.vstack((ecoli_CTR, ecoli_WA_total, empty_drums_graph))
# print('Intact drums has ' + str(intact_drums.shape[0]) + ' cases.')

## Merging of all cases of bacteria: All bacteria with and without antibiotics

In [99]:
bacteria_all = np.vstack((ecoli_CTR, ecoli_WA_total))

## Summary of import:

In [100]:
print('empty_drums_30: Empty cavities has ' + str(empty_drums_30.shape[0]) + ' cases.')
print('empty_drums_graph: Empty drums has ' + str(empty_drums_graph.shape[0]) + ' cases.')
print('ecoli_CTR: Control cases has ' + str(ecoli_CTR.shape[0]) + ' cases.')
print('ecoli_WA_subs: E. Coli susceptible has ' + str(ecoli_WA_subs.shape[0]) + ' cases.')
print('ecoli_WA_resis: E. Coli resistant has ' + str(ecoli_WA_resis.shape[0]) + ' cases.')
print('ecoli_WA_total: All E. Coli with antibiotics has ' + str(ecoli_WA_total.shape[0]) + ' cases.')
print('intact_drums: Intact drums has ' + str(intact_drums.shape[0]) + ' cases.')
print('bacteria_all: All E. Coli data has ' + str(bacteria_all.shape[0]) + ' cases.')
print('Therefore, all data has ' + str(intact_drums.shape[0] + empty_drums_30.shape[0]) + ' cases.')
print()
print('For cases with other bacteria:')
print()
print('kpneumoniae_KPC2_CTR: All Klabsiella Pneumoniae data has ' + str(kpneumoniae_KPC2_CTR.shape[0]) + ' cases.')
print('Paeruginosa_L1171_CTR: All Pseudomonas aeruginosa data has ' + str(Paeruginosa_L1171_CTR.shape[0]) + ' cases.')
print('abaumannii_B1380_CTR: All Acinetobacter Baumannii data has ' + str(abaumannii_B1380_CTR.shape[0]) + ' cases.')
print('S_aureus_CTR: All Staphylococcus Aureus data has ' + str(S_aureus_CTR.shape[0]) + ' cases.')
print('senteriditis_S1400_CTR: All Salmonella Enteritidis data has ' + str(senteriditis_S1400_CTR.shape[0]) + ' cases.')
print('Sagalactiae_CTR: All Streptococcus Agalactiae data has ' + str(Sagalactiae_CTR.shape[0]) + ' cases.')

empty_drums_30: Empty cavities has 828 cases.
empty_drums_graph: Empty drums has 928 cases.
ecoli_CTR: Control cases has 1130 cases.
ecoli_WA_subs: E. Coli susceptible has 380 cases.
ecoli_WA_resis: E. Coli resistant has 207 cases.
ecoli_WA_total: All E. Coli with antibiotics has 587 cases.
intact_drums: Intact drums has 2645 cases.
bacteria_all: All E. Coli data has 1717 cases.
Therefore, all data has 3473 cases.

For cases with other bacteria:

kpneumoniae_KPC2_CTR: All Klabsiella Pneumoniae data has 526 cases.
Paeruginosa_L1171_CTR: All Pseudomonas aeruginosa data has 95 cases.
abaumannii_B1380_CTR: All Acinetobacter Baumannii data has 106 cases.
S_aureus_CTR: All Staphylococcus Aureus data has 204 cases.
senteriditis_S1400_CTR: All Salmonella Enteritidis data has 198 cases.
Sagalactiae_CTR: All Streptococcus Agalactiae data has 100 cases.
